# Task 1
## Exploratory Data Analysis

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.loader import DataLoader
from src.data.validator import DatasetValidator
from src.eda.analyzer import EDAAnalyzer
from src.eda.visualizer import EDAVisualizer

In [ ]:
loader = DataLoader('../dataset/students_dataset.csv')
df = loader.load_data()
df.head()

In [ ]:
validator = DatasetValidator(df)

print("Shape:", validator.check_shape())
print("\nDtypes:\n", validator.check_dtypes())
print("\nDuplicates:", validator.check_duplicates())
print("\nMissing Values:\n", validator.check_missing_values())
print("\nTarget Distribution:\n", validator.class_distribution("num"))

In [ ]:
analyzer = EDAAnalyzer(df)

stats = analyzer.descriptive_statistics()
stats

In [ ]:
sns.countplot(x=df["num"])
plt.title("Target Distribution")
plt.show()

In [ ]:
visualizer = EDAVisualizer()

corr = analyzer.correlation_matrix()
visualizer.plot_correlation_heatmap(corr)

continuous_features = ["age", "trestbps", "chol", "thalach", "oldpeak"]
visualizer.plot_distributions(df, continuous_features, "num")

In [ ]:
# PCA
X = df.drop(columns=["num"])
y = df["num"]

# For now we simply drop missing values
X_clean = X.dropna() 
y_clean = df.loc[X_clean.index, "num"]

coords, variance = analyzer.run_pca(X_clean)
plt.figure(figsize=(8, 6))
sns.scatterplot(x=coords[:, 0], y=coords[:, 1], hue=y_clean)
plt.title(f"PCA (Variance: {sum(variance):.2%})")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

The PCA plot shows a significant amount of overlap between the two classes (no heart disease vs. presence of heart disease).
This suggests that a simple linear boundary may not be sufficient for high accuracy, and more complex, non-linear models will likely be necessary to capture the relationships in the data.

# Task 2
## Intuition & Preparation for the nCV Implementation

In [ ]:
from IPython.display import Image
Image(filename='Diagram.jpg') 

**Outer Loop (N=5):** The 242 patient records are split into 5 folds. One fold is set aside as the **Test Fold (unseen data)**, and the other four become the **Outer Training Sets**.  

**Inner Loop (K=3):** The Outer Training Set is split into 3 folds. This loop is mainly used for Hyperparameter Tuning and model selection. 

**Preprocessing Pipeline:** Inside each split, the data passes through:

- Imputation: To handle missing clinical values.  
- Scaling/Encoding: To prepare continuous and categorical features.  
- Feature Selection: To identify the most informative predictors.  
- Training: The algorithm is trained on the processed inner folds.  

**Winning Params & Evaluation:** The best hyperparameters found in the inner loop are used to train a model on the Outer Training Set. This model then makes predictions on the Test Fold.

Repeat...

# Task 3
## Nested Cross-Validation Implementation

In [ ]:
from src.cv.nested_cv import RepeatedNestedCV

In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Define the estimators
estimators = {
    'LR': LogisticRegression(penalty='elasticnet', solver='saga', max_iter=5000, random_state=42),
    'GNB': GaussianNB(),
    'LDA': LinearDiscriminantAnalysis(),
    'RF': RandomForestClassifier(random_state=42),
    'XGB': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    'LGBM': LGBMClassifier(random_state=42, verbose=-1),
    'CB': CatBoostClassifier(silent=True, random_state=42)
}

# Define hyperparameter spaces
param_grids = {
    'LR': {
        'clf__C': [0.1, 1, 10],
        'clf__l1_ratio': [0.2, 0.5, 0.8]
    },
    'GNB': {},
    'LDA': {
        'clf__solver': ['svd', 'lsqr']
    },
    'RF': {
        'clf__n_estimators': [100, 200],
        'clf__max_depth': [None, 10, 20],
        'clf__min_samples_split': [2, 5]
    },
    'XGB': {
        'clf__learning_rate': [0.01, 0.1],
        'clf__n_estimators': [100, 200],
        'clf__max_depth': [3, 6]
    },
    'LGBM': {
        'clf__learning_rate': [0.01, 0.1],
        'clf__num_leaves': [31, 50]
    },
    'CB': {
        'clf__depth': [4, 6],
        'clf__learning_rate': [0.01, 0.1]
    }
}

In [ ]:
cv = RepeatedNestedCV()

# --- Baseline Comparison (No Inner Loop) ---
empty_grids = {name: {} for name in estimators.keys()}
print("Starting Baseline Comparison...")
baseline_results = cv.run(X, y, estimators, empty_grids)

# --- Full nested CV (With Tuning) ---
print("\nStarting Full Nested CV (This may take a while)...")
full_ncv_results = cv.run(X, y, estimators, param_grids)

In [ ]:
summary_data = []

for name, df_results in full_ncv_results.items():
    # Calculate median and 95% Confidence Intervals (Bootstrap approach)
    medians = df_results.median()
    
    # Store for comparison
    row = {
        'Algorithm': name,
        'Median_MCC': medians['MCC'],
        'Median_AUC': medians['AUC'],
        'Median_BA': medians['BA'],
        'Median_F1': medians['F1']
    }
    summary_data.append(row)

# Create summary table
summary_df = pd.DataFrame(summary_data).sort_values(by='Median_MCC', ascending=False)
print("\n--- Final Performance Summary (Sorted by MCC) ---")
print(summary_df)